# Generate COVID Trajectories

Train multiple single-head flow-matching models with different random seeds, without covariates, generate general COVID severity trajectories from train/validation samples, and save one long-form CSV per replicate for downstream analysis.


## 1. Setup configs


In [ ]:
from pathlib import Path
import os
import sys
import tempfile
import warnings

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from pandas.errors import PerformanceWarning

PROJECT_ROOT = Path.cwd().parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "proteoflow-matplotlib"))
warnings.filterwarnings("ignore", category=PerformanceWarning)

from proteoflow.data.dataset import FlowMatchingDataModule
from proteoflow.data.load import load_covid
from proteoflow.models import FlowMatchingModel, sample_trajectories
from proteoflow.training.utils import set_seed

CONFIG = {
    "replicate_seeds": [42, 43, 44, 45, 46],
    "data_dir": PROJECT_ROOT / "data" / "covid",
    "output_dir": PROJECT_ROOT / "results" / "covid" / "generated_trajectories",
    "model_dir": PROJECT_ROOT / "results" / "covid" / "models",
    "test_size": 0.2,
    "covariates": [],
    "continuous_covariates": [],
    "n_pca_components": 30,
    "n_pairs": 3000,
    "time_tolerance": 1e-6,
    "step_sizes": (1, 2),
    "step_probabilities": (0.85, 0.15),
    "balance_groups": True,
    "balance_transitions": True,
    "spatial_weight": 0.1,
    "continuous_covariate_weight": 0.0,
    "continuous_covariate_bandwidth": 1.0,
    "batch_size": 64,
    "hidden_dim": 512,
    "n_blocks": 3,
    "dropout": 0.1,
    "covariate_embed_dim": 64,
    "time_embed_dim": 64,
    "n_epochs": 100,
    "lr": 1e-4,
    "patience": 10,
    "trajectory_t_start": 0,
    "trajectory_t_end": 1.0,
    "trajectory_steps": 50,
    "trajectory_replicates_per_start": 1,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

## 2. Create dataloader


In [5]:
def load_replicate_dataframe(seed: int) -> pd.DataFrame:
    df = load_covid(
        data_dir=str(CONFIG["data_dir"]),
        test_size=CONFIG["test_size"],
        random_state=seed,
        covariates=CONFIG["covariates"] + CONFIG["continuous_covariates"],
        annotate_proteins=True,
        fetch_protein_annotations=False,
    )
    df = df.copy()
    df.insert(0, "sample_id", df.index.astype(str))
    return df


def create_data_module(seed: int) -> FlowMatchingDataModule:
    set_seed(seed)
    df = load_replicate_dataframe(seed)
    data_module = FlowMatchingDataModule(
        df=df,
        covariate_cols=CONFIG["covariates"],
        continuous_covariate_cols=CONFIG["continuous_covariates"],
        n_pca_components=CONFIG["n_pca_components"],
        n_pairs=CONFIG["n_pairs"],
        time_tolerance=CONFIG["time_tolerance"],
        step_sizes=CONFIG["step_sizes"],
        step_probabilities=CONFIG["step_probabilities"],
        balance_groups=CONFIG["balance_groups"],
        balance_transitions=CONFIG["balance_transitions"],
        spatial_weight=CONFIG["spatial_weight"],
        continuous_covariate_weight=CONFIG["continuous_covariate_weight"],
        continuous_covariate_bandwidth=CONFIG["continuous_covariate_bandwidth"],
        batch_size=CONFIG["batch_size"],
        metadata_cols=["sample_id"],
    )
    data_module.setup()
    return data_module


## 3. Train model


In [6]:
def create_model(data_module: FlowMatchingDataModule) -> FlowMatchingModel:
    return FlowMatchingModel.single(
        input_dim=data_module.n_features,
        hidden_dim=CONFIG["hidden_dim"],
        n_blocks=CONFIG["n_blocks"],
        dropout=CONFIG["dropout"],
        covariate_embed_dim=CONFIG["covariate_embed_dim"],
        time_embed_dim=CONFIG["time_embed_dim"],
    )


def train_model(data_module: FlowMatchingDataModule, seed: int) -> tuple[FlowMatchingModel, dict]:
    set_seed(seed)
    flow_model = create_model(data_module)
    flow_model, metrics = flow_model.fit(
        data_module=data_module,
        covariates=CONFIG["covariates"],
        continuous_covariates=CONFIG["continuous_covariates"],
        n_epochs=CONFIG["n_epochs"],
        lr=CONFIG["lr"],
        patience=CONFIG["patience"],
    )
    return flow_model, metrics


def metrics_to_frame(metrics: dict, replicate: int, seed: int) -> pd.DataFrame:
    n_epochs = len(metrics["train_losses"])
    return pd.DataFrame({
        "replicate": replicate,
        "seed": seed,
        "epoch": np.arange(1, n_epochs + 1),
        "train_total_loss": metrics["train_losses"],
        "val_total_loss": metrics.get("val_losses", []) + [np.nan] * (n_epochs - len(metrics.get("val_losses", []))),
        "train_fit_loss": metrics.get("train_fit_losses", [np.nan] * n_epochs),
        "val_fit_loss": metrics.get("val_fit_losses", []) + [np.nan] * (n_epochs - len(metrics.get("val_fit_losses", []))),
    })
temp_model = create_model(create_data_module(42))
sum(p.numel() for p in temp_model.velocity_net.parameters())

Applying PCA: reducing from 94 to 30 dimensions.
Total explained variance: 0.8702024943120292
Applying PCA: reducing from 94 to 30 dimensions.
Total explained variance: 0.8702024943120292


2001182

## 4. Generate trajectories from train and validation starts


In [7]:
def start_sample_lookup(data_module: FlowMatchingDataModule, datasplit: str) -> pd.Series:
    starts = data_module.df[data_module.df["split"] == datasplit].copy()
    starts = starts[starts["time"] <= CONFIG["trajectory_t_start"]]
    starts = starts.reset_index(drop=True)
    return starts["sample_id"]


def trajectories_to_long(
    trajectory_results: dict,
    data_module: FlowMatchingDataModule,
    datasplit: str,
    replicate: int,
    seed: int,
) -> pd.DataFrame:
    abundances = trajectory_results["protein_abundances"]
    velocities = trajectory_results["protein_velocities"]
    if abundances is None or velocities is None or abundances.empty:
        return pd.DataFrame()

    id_cols = ["path_id", "start_idx", "rep", "step", "t", *CONFIG["covariates"], *CONFIG["continuous_covariates"]]
    id_cols = [col for col in id_cols if col in abundances.columns]
    protein_cols = data_module.protein_cols

    abundance_long = abundances.melt(
        id_vars=id_cols,
        value_vars=protein_cols,
        var_name="protein",
        value_name="abundance",
    )
    velocity_long = velocities.melt(
        id_vars=id_cols,
        value_vars=protein_cols,
        var_name="protein",
        value_name="velocity",
    )

    long_df = abundance_long.merge(
        velocity_long,
        on=id_cols + ["protein"],
        how="inner",
        validate="one_to_one",
    )
    sample_lookup = start_sample_lookup(data_module, datasplit)
    long_df["start_sample"] = long_df["start_idx"].map(sample_lookup)
    long_df["datasplit"] = datasplit
    long_df["replicate"] = replicate
    long_df["seed"] = seed
    long_df = long_df.rename(columns={"rep": "trajectory_replicate"})

    column_order = [
        "start_sample",
        "datasplit",
        "replicate",
        "seed",
        "path_id",
        "trajectory_replicate",
        "step",
        "t",
        "protein",
        "abundance",
        "velocity",
    ]
    return long_df[[col for col in column_order if col in long_df.columns]]


def generate_split_trajectories(
    data_module: FlowMatchingDataModule,
    flow_model: FlowMatchingModel,
    datasplit: str,
    replicate: int,
    seed: int,
) -> pd.DataFrame:
    trajectory_results = sample_trajectories(
        data_module,
        flow_model,
        split=datasplit,
        t_start_max=CONFIG["trajectory_t_start"],
        t_end=CONFIG["trajectory_t_end"],
        n_steps=CONFIG["trajectory_steps"],
        n_trajectories_per_start=CONFIG["trajectory_replicates_per_start"],
        pca_reducer=data_module.pca_reducer,
        protein_cols=data_module.protein_cols,
    )
    return trajectories_to_long(
        trajectory_results=trajectory_results,
        data_module=data_module,
        datasplit=datasplit,
        replicate=replicate,
        seed=seed,
    )


## 5. Generate and save replicate tables


In [ ]:
def replicate_output_path(replicate: int, seed: int) -> Path:
    return CONFIG["output_dir"] / f"replicate_{replicate:02d}_seed_{seed}.csv"


def model_checkpoint_path(replicate: int, seed: int) -> Path:
    return CONFIG["model_dir"] / f"replicate_{replicate:02d}_seed_{seed}.pt"


def save_model_checkpoint(
    flow_model: FlowMatchingModel,
    data_module: FlowMatchingDataModule,
    metrics: dict,
    replicate: int,
    seed: int,
) -> Path:
    checkpoint_path = model_checkpoint_path(replicate, seed)
    torch.save(
        {
            "velocity_state_dict": {
                key: value.detach().cpu()
                for key, value in flow_model.velocity_net.state_dict().items()
            },
            "model_config": {
                "input_dim": data_module.n_features,
                "hidden_dim": CONFIG["hidden_dim"],
                "n_blocks": CONFIG["n_blocks"],
                "dropout": CONFIG["dropout"],
                "covariate_embed_dim": CONFIG["covariate_embed_dim"],
                "time_embed_dim": CONFIG["time_embed_dim"],
            },
            "covariates": CONFIG["covariates"],
            "continuous_covariates": CONFIG["continuous_covariates"],
            "covariate_categories": data_module.covariate_categories,
            "continuous_covariate_stats": data_module.continuous_covariate_stats,
            "protein_cols": data_module.protein_cols,
            "feature_cols": data_module.feature_cols,
            "pca_reducer": data_module.pca_reducer,
            "metrics": metrics,
            "replicate": replicate,
            "seed": seed,
        },
        checkpoint_path,
    )
    return checkpoint_path


CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)
CONFIG["model_dir"].mkdir(parents=True, exist_ok=True)
saved_outputs = []
training_metric_tables = []

for replicate, seed in enumerate(CONFIG["replicate_seeds"], start=1):
    print(f"Replicate {replicate}/{len(CONFIG['replicate_seeds'])} | seed={seed}")
    data_module = create_data_module(seed)
    flow_model, metrics = train_model(data_module, seed)
    training_metric_tables.append(metrics_to_frame(metrics, replicate, seed))
    #checkpoint_path = save_model_checkpoint(flow_model, data_module, metrics, replicate, seed)
    #print(f"  saved model: {checkpoint_path}") dont save now

    replicate_tables = []
    for datasplit in ["train", "val"]:
        split_long = generate_split_trajectories(
            data_module=data_module,
            flow_model=flow_model,
            datasplit=datasplit,
            replicate=replicate,
            seed=seed,
        )
        print(f"  {datasplit}: {len(split_long):,} rows")
        replicate_tables.append(split_long)

    replicate_trajectories = pd.concat(replicate_tables, ignore_index=True)
    output_path = replicate_output_path(replicate, seed)
    #replicate_trajectories.to_csv(output_path, index=False) dont save now
    saved_outputs.append({
        "replicate": replicate,
        "seed": seed,
        "path": str(output_path),
        "model_path": str(checkpoint_path),
        "n_rows": len(replicate_trajectories),
        "n_columns": len(replicate_trajectories.columns),
    })
    print(f"  saved: {output_path}")

training_metrics = pd.concat(training_metric_tables, ignore_index=True)
saved_outputs = pd.DataFrame(saved_outputs)

display(saved_outputs)
display(replicate_trajectories.head())


Replicate 1/5 | seed=42
Applying PCA: reducing from 94 to 30 dimensions.
Total explained variance: 0.8702024943120292
Applying PCA: reducing from 94 to 30 dimensions.
Total explained variance: 0.8702024943120292
Epoch 1/100 | Train: 2032.2781 | Val: 1992.8579
Epoch 11/100 | Train: 1874.5319 | Val: 1938.8545
Epoch 21/100 | Train: 1814.3053 | Val: 1945.1455
Early stopping at epoch 30
  saved model: /Users/erikhartman/dev/flow-matching/results/covid/models/replicate_01_seed_42.pt
  train: 570,486 rows
  val: 148,614 rows
  saved: /Users/erikhartman/dev/flow-matching/results/covid/generated_trajectories_v3/replicate_01_seed_42.csv
Replicate 2/5 | seed=43
Applying PCA: reducing from 94 to 30 dimensions.
Total explained variance: 0.8700866165622462
Applying PCA: reducing from 94 to 30 dimensions.
Total explained variance: 0.8700866165622462
Epoch 1/100 | Train: 2040.2124 | Val: 2018.0584
Epoch 11/100 | Train: 1906.8363 | Val: 1999.2726
Early stopping at epoch 15
  saved model: /Users/erikhar

,replicate,seed,path,model_path,n_rows,n_columns
0,1,42,/Users/erikhartman/dev/flow-matching/results/c...,/Users/erikhartman/dev/flow-matching/results/c...,719100,11
1,2,43,/Users/erikhartman/dev/flow-matching/results/c...,/Users/erikhartman/dev/flow-matching/results/c...,719100,11
2,3,44,/Users/erikhartman/dev/flow-matching/results/c...,/Users/erikhartman/dev/flow-matching/results/c...,719100,11
3,4,45,/Users/erikhartman/dev/flow-matching/results/c...,/Users/erikhartman/dev/flow-matching/results/c...,719100,11
4,5,46,/Users/erikhartman/dev/flow-matching/results/c...,/Users/erikhartman/dev/flow-matching/results/c...,719100,11


,start_sample,datasplit,replicate,seed,path_id,trajectory_replicate,step,t,protein,abundance,velocity
0,520_TOF1_AF_027_ZeBanC_P1_B10,train,5,46,0,0,0,0.00,ALB,16.401627,-0.774884
1,520_TOF1_AF_027_ZeBanC_P1_B10,train,5,46,0,0,1,0.02,ALB,16.386617,-0.727736
2,520_TOF1_AF_027_ZeBanC_P1_B10,train,5,46,0,0,2,0.04,ALB,16.372521,-0.683592
3,520_TOF1_AF_027_ZeBanC_P1_B10,train,5,46,0,0,3,0.06,ALB,16.359275,-0.642612
4,520_TOF1_AF_027_ZeBanC_P1_B10,train,5,46,0,0,4,0.08,ALB,16.346816,-0.604791


In [9]:
training_metrics.to_csv(CONFIG["output_dir"] / "training_metrics.csv", index=False)
